# Imports

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools import add_constant
import plotly.graph_objects as go

# Constants

## Control

In [ ]:
FETCH_DATA = False
SAVE_RESULT = True
PLOT_RESULT = False

## Dataset

In [4]:
START_DATE = "2024-12-01"
END_DATE = "2025-12-01"

TOP_50 = ["BTC", "ETH", "BNB", "XRP", "SOL", "TRX", "DOGE", "ADA", "BCH", "LINK", "RAIN", "XMR", "XLM", "ZEC", "LEO", "LTC", "DAI", "SUI", "AVAX", "HBAR", "SHIB", "NIGHT", "TON", "CRO", "UNI", "DOT", "AAVE", "CC", "BGB", "ASTER", "PI", "ENA", "SKY", "KCS", "WLD", "ONDO", "KAS", "APT", "ARB", "ALGO", "FLR", "ATOM", "FIL", "QNT", "VET", "SET", "M", "CBBTC", "WBT", "PYUSD"]

## Routes

In [5]:
TICKERS_DIR = "tickers"
PLOTS_DIR = "plots"

# Configuration

# Functions

In [6]:
def fetch_from_yfinance(ticker: str, route: str, start_date, end_date, fetch: bool=True):
    if fetch:
        s = ticker.upper().strip()
        yahoo_format = s if s.endswith("-USD") else f"{s}-USD"
        
        df1 = yf.download(
            tickers=yahoo_format,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        df2 = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        
        if len(df1) > 0:
            df = df1
        if len(df2) > 0:
            df = df2
            
        df.to_csv(route)
        df = pd.read_csv(route, skiprows=[1, 2], header=0)
        df = df.rename(columns={'Price': 'Date'})
        df = df.set_index('Date')
        df.to_csv(route)

    else:
        df = pd.read_csv(route)
    return df

In [7]:
def adf_test(ticker, df, column='Close'):
    if df is None or df.empty or column not in df.columns:
        return {"Ticker": ticker, "Error": "Missing Data"}

    series = df[column].dropna()
    
    if len(series) < 20:
        return {"Ticker": ticker, "Error": f"Insufficient data points: {len(series)}"}

    result = adfuller(series, autolag='AIC')
    
    adf_output = {
        "Ticker": ticker,
        "ADF Statistic": round(result[0], 4),
        "p-value": round(result[1], 4),
        "Stationary": result[1] < 0.05,
        "Lags Used": result[2],
        "Observations": result[3]
    }
    
    return adf_output

In [8]:
def hurst_exponent(series, q=2.0, max_lag=None, min_lag=2):
    series = np.asarray(series, dtype=float)
    n = len(series)

    if max_lag is None:
        max_lag = n // 4
    if max_lag <= min_lag:
        raise ValueError("max_lag must be > min_lag")

    # Lags
    lags = np.arange(min_lag, max_lag)

    # K_q values for each lag
    K = np.zeros_like(lags, dtype=float)

    for i, lag in enumerate(lags):
        diffs = np.abs(series[lag:] - series[:-lag])
        K[i] = np.mean(diffs ** q)

    # Fit log–log to estimate slope
    log_lags = np.log(lags)
    log_K = np.log(K)

    slope, _ = np.polyfit(log_lags, log_K, 1)

    Hurst = slope / q
    
    return Hurst

In [9]:
def half_life(series):
    series = series.dropna()
    
    # Create lagged series
    price_lag = series.shift(1)
    price_diff = series - price_lag
    
    # Remove NaN values
    valid_data = pd.concat([price_lag, price_diff], axis=1).dropna()
    price_lag_clean = valid_data.iloc[:, 0]
    price_diff_clean = valid_data.iloc[:, 1]
    
    # Add constant for regression
    X = add_constant(price_lag_clean)
    
    # Perform OLS regression
    try:
        model = OLS(price_diff_clean, X)
        results = model.fit()
        beta = results.params[1]
        
        # Calculate half-life
        half_life = -np.log(2) / np.log(1 + beta) if (1 + beta) > 0 else np.nan
        print('Half-life: %0.2f' % half_life)
        return half_life
    except:
        return np.nan

In [10]:
def store_series_plot(df, ticker: str, route: str = PLOTS_DIR):
    if not os.path.exists(route):
        os.makedirs(route)
        
    fig = go.Figure(data=[go.Candlestick(
        x=df['Date'],
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
    )])

    fig.update_layout(
        xaxis_title='Date',
        yaxis_title='Price (USDT)',
        xaxis_rangeslider_visible=False,   
        template='plotly_white',
        height=500,
        width=900,
        showlegend=False
    )
    
    save_path = os.path.join(route, f"{ticker}.png")
    fig.write_image(save_path)

# Fetch data

## Fetch merket data of top 50 crypto

In [ ]:
dfs = {}
if not os.path.exists(TICKERS_DIR):
    os.makedirs(TICKERS_DIR)
    print(f"Created directory: {TICKERS_DIR}")
for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        dfs[ticker] = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
        if PLOT_RESULT:
            store_series_plot(dfs[ticker], ticker)
    except Exception as e:
        print(f"Failed to fetch {ticker}: {e}")

# Calculate parameters

In [12]:
summary_results = []

for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        df = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
        res = adf_test(ticker, df)
        res['hurst'] = hurst_exponent(df['Close'])
        res['half_life'] = half_life(df['Close'])
        summary_results.append(res)
        
    except Exception as e:
        print(f"Failed {ticker}: {e}")

Failed XRP: max_lag must be > min_lag
Failed SET: max_lag must be > min_lag


In [13]:
summary_df = pd.DataFrame(summary_results)
summary_df.head()

,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,hurst,half_life
0,BTC,-1.7554,0.4028,False,0,248,0.447950,NaN
1,ETH,-1.2146,0.6673,False,1,247,0.611152,NaN
2,BNB,-1.0345,0.7405,False,8,356,0.466865,NaN
3,SOL,-2.1083,0.2412,False,1,247,0.485505,NaN
4,TRX,-0.1726,0.9417,False,12,236,0.538031,NaN


## Filter tickers with p-value $\le$ 0.05

In [14]:
stationary_95_df = summary_df[summary_df['p-value'] <= 0.05].copy()
stationary_95_df = stationary_95_df.sort_values(by='p-value')
print("Stationary tickers:", len(stationary_95_df))
stationary_95_df.head()

Stationary tickers: 11


,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,hurst,half_life
15,DAI,-5.8488,0.0000,True,3,361,0.017397,NaN
47,PYUSD,-7.3310,0.0000,True,2,362,0.000187,NaN
24,DOT,-4.6399,0.0001,True,6,358,0.461340,NaN
16,SUI,-4.5019,0.0002,True,0,248,0.199470,NaN
14,LTC,-4.1974,0.0007,True,0,248,0.143460,NaN
